# Q-SAR Sentinel 2.0 — Bi-Temporal Hybrid Quantum-Classical Training (Google Colab)

**Challenge:** 2026 NASA International Space Apps Challenge — *"Dancing with the SARs"*  
**Architecture:** Bi-Temporal 4-Channel Input `(Post Co, Post Cross, tanh(2.4 · ΔCo), tanh(2.4 · ΔCross))` → Dual-Stream `SARConvStem` (`3×3` Spatial + `AvgPool2d(4)` Radiometric Branch → 4 Angles) → **4-Qubit PennyLane Variational Circuit** (`AngleEmbedding(Y)` + 2× `StronglyEntanglingLayers`, **8 Observables**: `4× ⟨Zᵢ⟩ + 4× ⟨ZᵢZᵢ₊₁⟩`) → `Linear(8, 4)` Readout (`9,732` total params, `~45 KB` checkpoint).

### Workflow
1. Install dependencies & optionally query live NASA Earthdata CMR (`NISAR_L2_GCOV_PROVISIONAL_V1` / `OPERA_L2_RTC-S1_V1`).
2. Generate paired **Pre + Post** SAR physics data across normal baseline/rivers, open-water + double-bounce floods, wildfire burn scars, and smallholder mosaic crop stress.
3. Train `HybridQSARNet` alongside `ClassicalSARNet` (zero real-scene training leakage), then evaluate on `L=4`, `L=1` speckle stress, cross-domain shift, and few-shot subsets.
4. Export `<5 MB` `qsar_hybrid_weights.pt` (`weight_source="bitemporal_hybrid_trained"`) and `benchmark_metrics.json` for `backend/models/`.

In [ ]:
!pip install -q pennylane pennylane-lightning torch torchvision earthaccess h5py rasterio scikit-learn matplotlib

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pennylane as qml
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | PennyLane {qml.__version__} | Device: {DEVICE}")

In [ ]:
# Optional: Live NASA Earthdata CMR discovery for Nagpur Flood, California Wildfire & Vidarbha ROIs
ENABLE_NASA_AUTH = False  # Set True if you want interactive Earthdata Login to download full .h5/.tif files

try:
    import earthaccess

    if ENABLE_NASA_AUTH:
        earthaccess.login(strategy="interactive")

    for name, bbox, t_win in [
        ("Nagpur / Wainganga Flood ROI", (79.02, 20.92, 79.26, 21.16), ("2026-06-01", "2026-09-01")),
        ("California Wildfire ROI", (-121.62, 39.72, -121.38, 39.96), ("2026-07-01", "2026-10-01")),
    ]:
        for col_name in ("OPERA_L2_RTC-S1_V1", "NISAR_L2_GCOV_PROVISIONAL_V1", "NISAR_UR_L2"):
            hits = earthaccess.search_data(short_name=col_name, bounding_box=bbox, temporal=t_win, count=2)
            if hits:
                print(f"[NASA CMR] {name} -> {col_name}: found {len(hits)} sample granules")
                break
except Exception as exc:
    print(f"NASA CMR search skipped ({exc}); continuing with calibrated bi-temporal physics training.")

In [ ]:
CLASS_NAMES = {
    0: "Normal Terrain",
    1: "Flood Inundation",
    2: "Wildfire Burn Scar",
    3: "Crop/Soil Stress",
}
HH_DB_MIN, HH_DB_MAX = -30.0, 2.0
HV_DB_MIN, HV_DB_MAX = -35.0, -5.0


def db_to_norm(db_val, db_min, db_max):
    return np.clip((db_val - db_min) / (db_max - db_min), 0.0, 1.0).astype(np.float32)


def _apply_gamma_speckle(db_clean, look_count, rng):
    lin_clean = np.power(10.0, db_clean / 10.0)
    looks = max(1, int(look_count))
    speckle = rng.gamma(shape=float(looks), scale=1.0 / float(looks), size=db_clean.shape)
    return (10.0 * np.log10(np.maximum(lin_clean * speckle, 1e-6))).astype(np.float32)


def _fbm_field(h, w, rng):
    yy, xx = np.meshgrid(np.linspace(0.0, 1.0, h), np.linspace(0.0, 1.0, w), indexing="ij")
    field = np.zeros((h, w), dtype=np.float32)
    for freq, amp in ((3.0, 0.55), (7.0, 0.30), (15.0, 0.15)):
        px, py = rng.uniform(0, 2 * np.pi, size=2)
        field += amp * np.sin(2 * np.pi * freq * xx + px) * np.cos(2 * np.pi * freq * yy + py)
    return field


def fuse_bitemporal_numpy(pre_np, post_np):
    """Matches backend.quantum_model.build_bitemporal_input exact 4-channel tensor."""
    delta = 0.5 + 0.5 * np.tanh(2.4 * (post_np - pre_np))
    return np.concatenate([post_np, delta], axis=1).astype(np.float32)


def generate_bitemporal_sar_dataset(num_samples=720, looks=(1, 2, 4, 8), seed=42, domain_shift=False):
    rng = np.random.default_rng(seed)
    pre_arr = np.zeros((num_samples, 2, 64, 64), dtype=np.float32)
    post_arr = np.zeros((num_samples, 2, 64, 64), dtype=np.float32)
    labels = np.zeros((num_samples,), dtype=np.int64)
    yy, xx = np.meshgrid(np.linspace(0, 1, 64), np.linspace(0, 1, 64), indexing="ij")
    shift_db = rng.uniform(-1.5, 1.5) if domain_shift else 0.0

    for i in range(num_samples):
        cls = i % 4
        labels[i] = cls
        tex = _fbm_field(64, 64, rng)
        look_count = int(looks[i % len(looks)])

        # Pre-event baseline (including permanent river channels in 25% of Class 0)
        pre_hh = rng.uniform(-11.8, -8.4) + 1.5 * tex + shift_db
        pre_hv = rng.uniform(-15.8, -12.0) + 1.3 * tex + shift_db

        if cls == 0:  # Stable terrain OR unchanged permanent river channel
            if (i // 4) % 4 == 0:
                river_mask = np.abs(xx - 0.5 + 0.1 * np.sin(4 * yy)) < 0.22
                pre_hh = np.where(river_mask, rng.uniform(-24.5, -20.5), pre_hh)
                pre_hv = np.where(river_mask, rng.uniform(-31.0, -27.5), pre_hv)
            post_hh = pre_hh + rng.uniform(-0.8, 0.8)
            post_hv = pre_hv + rng.uniform(-0.8, 0.8)
        elif cls == 1:  # Active Flood Inundation OR emergent double-bounce vegetation
            subtype = (i // 4) % 3
            if subtype == 0:  # Open-water inundation
                post_hh = rng.uniform(-25.5, -20.5) + 1.0 * tex + shift_db
                post_hv = rng.uniform(-32.0, -27.5) + 1.0 * tex + shift_db
            elif subtype == 1:  # Flooded-vegetation double-bounce (dCo - dCr > 1.8)
                post_hh = pre_hh + rng.uniform(3.2, 6.2)
                post_hv = pre_hv + rng.uniform(-0.8, 0.5)
            else:  # Mixed bank flooding
                split = (xx * rng.uniform(0.6, 1.0) + 0.15 * tex) > 0.45
                post_hh = np.where(split, pre_hh + rng.uniform(3.0, 5.5), rng.uniform(-24.5, -20.5) + shift_db)
                post_hv = np.where(split, pre_hv, rng.uniform(-31.0, -27.5) + shift_db)
        elif cls == 2:  # Wildfire Burn Scar (sharp cross-pol volume drop without standing water)
            post_hh = pre_hh + rng.uniform(-2.8, -0.5) + 1.2 * tex
            post_hv = pre_hv - rng.uniform(5.0, 10.5) + 1.0 * tex
        else:  # Smallholder Crop / Soil Moisture Stress mosaic
            mosaic = (np.sin(2 * np.pi * rng.uniform(3.0, 6.0) * xx) * np.cos(2 * np.pi * rng.uniform(3.0, 6.0) * yy)) > 0
            d_co = np.where(mosaic, rng.uniform(-4.5, -2.5), rng.uniform(2.2, 4.0))
            d_cr = np.where(mosaic, rng.uniform(-4.8, -2.4), rng.uniform(-2.8, 2.5))
            post_hh = pre_hh + d_co
            post_hv = pre_hv + d_cr

        pre_arr[i, 0] = db_to_norm(_apply_gamma_speckle(pre_hh, look_count, rng), HH_DB_MIN, HH_DB_MAX)
        pre_arr[i, 1] = db_to_norm(_apply_gamma_speckle(pre_hv, look_count, rng), HV_DB_MIN, HV_DB_MAX)
        post_arr[i, 0] = db_to_norm(_apply_gamma_speckle(post_hh, look_count, rng), HH_DB_MIN, HH_DB_MAX)
        post_arr[i, 1] = db_to_norm(_apply_gamma_speckle(post_hv, look_count, rng), HV_DB_MIN, HV_DB_MAX)

    return fuse_bitemporal_numpy(pre_arr, post_arr), labels


X_train_full, y_train_full = generate_bitemporal_sar_dataset(720, looks=(1, 2, 4, 8), seed=42)
X_val, y_val = generate_bitemporal_sar_dataset(240, looks=(4,), seed=999)
X_noisy_val, y_noisy_val = generate_bitemporal_sar_dataset(240, looks=(1,), seed=1337)
X_shift_val, y_shift_val = generate_bitemporal_sar_dataset(240, looks=(4,), seed=2027, domain_shift=True)
print(f"Train: {X_train_full.shape} | Val L=4: {X_val.shape} | Val L=1: {X_noisy_val.shape} | Shift: {X_shift_val.shape}")

In [ ]:
IN_CHANNELS = 4
N_QUBITS = 4
N_LAYERS = 2
N_Q_OBS = 8
N_CLASSES = 4

qdev = qml.device("default.qubit", wires=N_QUBITS)


@qml.qnode(qdev, interface="torch", diff_method="backprop")
def q_classify_qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)] + [
        qml.expval(qml.PauliZ(i) @ qml.PauliZ((i + 1) % N_QUBITS))
        for i in range(N_QUBITS)
    ]


class SARConvStem(nn.Module):
    """Dual-stream (Spatial 3x3 + AvgPool2d(4) Multi-Look) feature stem matching backend/quantum_model.py."""
    def __init__(self, in_channels=IN_CHANNELS, n_qubits=N_QUBITS):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AvgPool2d(kernel_size=2),
            nn.Conv2d(16, 24, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AvgPool2d(kernel_size=2),
            nn.Conv2d(24, 24, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.multilook_blocks = nn.Sequential(
            nn.AvgPool2d(kernel_size=4),
            nn.Conv2d(in_channels, 24, kernel_size=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.flatten = nn.Flatten()
        self.proj = nn.Linear(48, n_qubits)
        self.angle_bound = nn.Tanh()

    def forward_features(self, x):
        h_spatial = self.flatten(self.conv_blocks(x))
        h_radiom = self.flatten(self.multilook_blocks(x))
        return torch.cat([h_spatial, h_radiom], dim=-1)

    def forward(self, x):
        return self.angle_bound(self.proj(self.forward_features(x))) * float(np.pi)


class HybridQSARNet(nn.Module):
    """Matches backend/quantum_model.py HybridQSARNet state_dict keys strictly."""
    def __init__(self, in_channels=IN_CHANNELS, n_qubits=N_QUBITS, n_layers=N_LAYERS):
        super().__init__()
        self.stem = SARConvStem(in_channels=in_channels, n_qubits=n_qubits)
        self.q_layer = qml.qnn.TorchLayer(
            q_classify_qnode,
            weight_shapes={"weights": (n_layers, n_qubits, 3)},
        )
        self.classifier_head = nn.Linear(N_Q_OBS, N_CLASSES)
        self.classical_head = nn.Sequential(
            nn.Linear(n_qubits, 8),
            nn.GELU(),
            nn.Linear(8, N_CLASSES),
        )

    def forward(self, x):
        angles = self.stem(x)
        q_expvals = self.q_layer(angles)
        q_logits = self.classifier_head(q_expvals)
        c_logits = self.classical_head(angles / float(np.pi))
        return q_logits, c_logits, angles, q_expvals


class ClassicalSARNet(nn.Module):
    """Standalone purely classical CNN baseline for fair evaluation."""
    def __init__(self, in_channels=IN_CHANNELS, n_qubits=N_QUBITS):
        super().__init__()
        self.stem = SARConvStem(in_channels=in_channels, n_qubits=n_qubits)
        self.head = nn.Sequential(
            nn.Linear(n_qubits, 8),
            nn.GELU(),
            nn.Linear(8, N_CLASSES),
        )

    def forward(self, x):
        return self.head(self.stem(x) / float(np.pi))


print("Hybrid params:", sum(p.numel() for p in HybridQSARNet().parameters()))
print("Classical params:", sum(p.numel() for p in ClassicalSARNet().parameters()))

In [ ]:
def evaluate_model(model, X_np, y_np, is_hybrid=False):
    model.eval()
    with torch.no_grad():
        xb = torch.from_numpy(X_np).to(DEVICE)
        out = model(xb)
        logits = out[0] if is_hybrid else out
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
    acc = float(np.mean(preds == y_np))
    macro_f1 = float(f1_score(y_np, preds, average="macro"))
    return acc, macro_f1, preds


def train_model(model, X_np, y_np, epochs=15, batch_size=16, lr=6e-3, is_hybrid=False, seed=42):
    torch.manual_seed(seed)
    model = model.to(DEVICE)
    ds = TensorDataset(torch.from_numpy(X_np), torch.from_numpy(y_np))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_acc": [], "val_f1": []}
    best_acc = -1.0
    best_state = None
    for ep in range(epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            if is_hybrid:
                q_logits, c_logits, _, _ = model(xb)
                loss = crit(q_logits, yb) + 0.5 * crit(c_logits, yb)
            else:
                loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            running_loss += float(loss.item()) * xb.size(0)
        sched.step()
        val_acc, val_f1, _ = evaluate_model(model, X_val, y_val, is_hybrid=is_hybrid)
        if val_acc >= best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        history["train_loss"].append(round(running_loss / len(ds), 4))
        history["val_acc"].append(round(val_acc, 4))
        history["val_f1"].append(round(val_f1, 4))
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


t0 = time.time()
hybrid_model, hybrid_hist = train_model(HybridQSARNet(), X_train_full, y_train_full, epochs=15, is_hybrid=True, seed=42)
classical_model, classical_hist = train_model(ClassicalSARNet(), X_train_full, y_train_full, epochs=15, is_hybrid=False, seed=42)
print(f"15-Epoch Bi-Temporal Benchmark Finished in {time.time() - t0:.1f}s")
print(f"Final L=4 Val -> Hybrid: {hybrid_hist['val_acc'][-1]:.4f} (F1: {hybrid_hist['val_f1'][-1]:.4f}) | "
      f"Classical: {classical_hist['val_acc'][-1]:.4f} (F1: {classical_hist['val_f1'][-1]:.4f})")

In [ ]:
# Few-Shot Small-Data & L=1 Speckle Stress Ablation
few_shot_budgets = [40, 80, 160, 400]
few_shot_curves = []

for n_sub in few_shot_budgets:
    X_sub, y_sub = X_train_full[:n_sub], y_train_full[:n_sub]
    bs = 8 if n_sub <= 80 else 16
    c_sub, _ = train_model(ClassicalSARNet(), X_sub, y_sub, epochs=15, batch_size=bs, lr=7e-3, is_hybrid=False, seed=42)
    q_sub, _ = train_model(HybridQSARNet(), X_sub, y_sub, epochs=15, batch_size=bs, lr=7e-3, is_hybrid=True, seed=42)
    c_acc, c_f1, _ = evaluate_model(c_sub, X_val, y_val, is_hybrid=False)
    q_acc, q_f1, _ = evaluate_model(q_sub, X_val, y_val, is_hybrid=True)
    c_noisy_acc, _, _ = evaluate_model(c_sub, X_noisy_val, y_noisy_val, is_hybrid=False)
    q_noisy_acc, _, _ = evaluate_model(q_sub, X_noisy_val, y_noisy_val, is_hybrid=True)
    row = {
        "n_train": n_sub,
        "classical_acc": round(c_acc, 4),
        "quantum_acc": round(q_acc, 4),
        "classical_f1": round(c_f1, 4),
        "quantum_f1": round(q_f1, 4),
        "classical_1look_acc": round(c_noisy_acc, 4),
        "quantum_1look_acc": round(q_noisy_acc, 4),
    }
    few_shot_curves.append(row)
    print(row)

In [ ]:
# Comparison Plots (Validation Curves, Few-Shot Generalization & Confusion Matrix)
plt.style.use("dark_background")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

epochs_x = np.arange(1, 16)
axes[0].plot(epochs_x, classical_hist["val_acc"], "o--", color="#38bdf8", label="Classical CNN")
axes[0].plot(epochs_x, hybrid_hist["val_acc"], "s-", color="#34d399", label="Hybrid Q-SAR (8-Obs)")
axes[0].set_title("15-Epoch Bi-Temporal Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.4, 1.02)
axes[0].legend()
axes[0].grid(alpha=0.25)

ns = [r["n_train"] for r in few_shot_curves]
axes[1].plot(ns, [r["classical_acc"] for r in few_shot_curves], "o--", color="#38bdf8", label="Classical CNN (L=4)")
axes[1].plot(ns, [r["quantum_acc"] for r in few_shot_curves], "s-", color="#34d399", label="Hybrid Q-SAR (L=4)")
axes[1].plot(ns, [r["quantum_1look_acc"] for r in few_shot_curves], "^:", color="#f59e0b", label="Hybrid Q-SAR (L=1 speckle)")
axes[1].set_title("Small-Data & Speckle Generalization")
axes[1].set_xlabel("Labeled Training Patches (N)")
axes[1].set_ylabel("Validation Accuracy")
axes[1].set_ylim(0.4, 1.02)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)

q_final_acc, q_final_f1, q_preds = evaluate_model(hybrid_model, X_val, y_val, is_hybrid=True)
c_final_acc, c_final_f1, _ = evaluate_model(classical_model, X_val, y_val, is_hybrid=False)
cm = confusion_matrix(y_val, q_preds)
axes[2].imshow(cm, cmap="viridis")
axes[2].set_title(f"Hybrid Q-SAR Confusion Matrix (Acc={q_final_acc:.2%})")
axes[2].set_xticks(range(4))
axes[2].set_yticks(range(4))
axes[2].set_xticklabels(["Norm", "Flood", "Fire", "Crop"], rotation=20)
axes[2].set_yticklabels(["Norm", "Flood", "Fire", "Crop"])
for r in range(4):
    for c in range(4):
        axes[2].text(c, r, str(cm[r, c]), ha="center", va="center", color="white")

plt.tight_layout()
plt.savefig("qsar_nasa_benchmarks.png", dpi=150)
plt.show()
print(classification_report(y_val, q_preds, target_names=[CLASS_NAMES[i] for i in range(4)]))

In [ ]:
# Export <5MB weights (`qsar_hybrid_weights.pt`) + bi-temporal benchmark JSON
weights_file = Path("qsar_hybrid_weights.pt")
bench_file = Path("benchmark_metrics.json")

hybrid_cpu = hybrid_model.cpu().eval()
torch.save(
    {
        "state_dict": hybrid_cpu.state_dict(),
        "weight_source": "bitemporal_hybrid_trained",
        "bitemporal_version": 2,
        "in_channels": IN_CHANNELS,
        "n_qubits": N_QUBITS,
        "n_layers": N_LAYERS,
        "n_q_obs": N_Q_OBS,
        "class_names": CLASS_NAMES,
    },
    weights_file,
)

size_kb = weights_file.stat().st_size / 1024.0
assert size_kb < 5120.0, f"Weight file too large: {size_kb:.1f} KB"

# Strict state_dict compatibility sanity check
reloaded = HybridQSARNet()
payload = torch.load(weights_file, map_location="cpu", weights_only=False)
reloaded.load_state_dict(payload["state_dict"], strict=True)

q_1look_acc, q_1look_f1, _ = evaluate_model(hybrid_cpu, X_noisy_val, y_noisy_val, is_hybrid=True)
c_1look_acc, c_1look_f1, _ = evaluate_model(classical_model.cpu(), X_noisy_val, y_noisy_val, is_hybrid=False)
q_shift_acc, q_shift_f1, _ = evaluate_model(hybrid_cpu, X_shift_val, y_shift_val, is_hybrid=True)
c_shift_acc, c_shift_f1, _ = evaluate_model(classical_model.cpu(), X_shift_val, y_shift_val, is_hybrid=False)

benchmark_payload = {
    "bitemporal_version": 2,
    "source": "colab_15_epoch_bitemporal_benchmark",
    "eval_samples": int(len(y_val)),
    "quantum_accuracy": round(q_final_acc, 4),
    "classical_accuracy": round(c_final_acc, 4),
    "quantum_macro_f1": round(q_final_f1, 4),
    "classical_macro_f1": round(c_final_f1, 4),
    "quantum_1look_accuracy": round(q_1look_acc, 4),
    "classical_1look_accuracy": round(c_1look_acc, 4),
    "quantum_shift_accuracy": round(q_shift_acc, 4),
    "classical_shift_accuracy": round(c_shift_acc, 4),
    "quantum_params": int(hybrid_cpu.q_layer.weights.numel() + sum(p.numel() for p in hybrid_cpu.classifier_head.parameters())),
    "total_params": int(sum(p.numel() for p in hybrid_cpu.parameters())),
    "weights_size_kb": round(size_kb, 2),
    "confusion_matrix": cm.tolist(),
    "few_shot_curves": few_shot_curves,
}
bench_file.write_text(json.dumps(benchmark_payload, indent=2), encoding="utf-8")
print(f"Exported {weights_file} ({size_kb:.1f} KB) and {bench_file}!")

try:
    from google.colab import files
    files.download(str(weights_file))
    files.download(str(bench_file))
    files.download("qsar_nasa_benchmarks.png")
except ImportError:
    print("Not running inside Google Colab; artifacts saved locally in current directory.")